# 🧪 Notebook de Prueba: Enriquecimiento IA de Actividades

Este notebook permite probar paso a paso el enriquecimiento de actividades con campos de IA:
- `es_aire_libre`: Clasificación booleana
- `edad_minima`: Extracción de edad mínima
- `edad_maxima`: Extracción de edad máxima

## Requisitos previos
1. Instalar dependencias: `pip install -r requirements-ia.txt`
2. Tener el archivo `actividades_procesadas.json` en la carpeta `data/`

## Paso 1: Configuración e Imports

In [ ]:
import sys
import os
import json
import time

# Añadir scripts al path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

from models.zero_shot import AirClassifier
from models.qa_age import AgeExtractor
from utils.text_processor import clean_text

print("✅ Imports cargados correctamente")

## Paso 2: Cargar actividades de prueba

In [ ]:
# Ruta al archivo de actividades
DATA_PATH = os.path.join('..', 'data', 'actividades_procesadas.json')

# Cargar actividades
if os.path.exists(DATA_PATH):
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        actividades = json.load(f)
    print(f"✅ Cargadas {len(actividades)} actividades")
    
    # Mostrar primera actividad como ejemplo
    if actividades:
        print("\n📋 Primera actividad:")
        print(f"   Título: {actividades[0].get('title', 'N/A')}")
        print(f"   Categoría: {actividades[0].get('category', 'N/A')}")
        print(f"   Descripción: {actividades[0].get('description', 'N/A')[:100]}...")
else:
    print(f"❌ No se encontró el archivo: {DATA_PATH}")
    print("   Asegúrate de tener el archivo en la carpeta data/")

## Paso 3: Seleccionar actividades de prueba

Seleccionamos un subconjunto pequeño para probar rápidamente

In [ ]:
# Número de actividades a probar
N_PRUEBA = 5

# Seleccionar primeras N actividades
actividades_prueba = actividades[:N_PRUEBA]

print(f"🧪 Actividades seleccionadas para prueba: {len(actividades_prueba)}")
for i, act in enumerate(actividades_prueba, 1):
    print(f"\n{i}. {act.get('title', 'Sin título')}")
    print(f"   Desc: {act.get('description', 'Sin descripción')[:80]}...")

## Paso 4: Inicializar modelos de IA

In [ ]:
print("🤖 Inicializando modelos...")
print("   Esto puede tardar 1-2 minutos la primera vez (descarga de modelos)")

# Inicializar clasificador de aire libre
air_classifier = AirClassifier(device="cpu")

# Inicializar extractor de edades
age_extractor = AgeExtractor(device="cpu")

print("\n✅ Modelos cargados y listos")

## Paso 5: Probar clasificación "es_aire_libre"

In [ ]:
print("🌳 Probando clasificación de 'aire libre'...\n")

for act in actividades_prueba:
    title = act.get('title', '')
    description = act.get('description', '')
    
    # Limpiar texto
    clean_desc = clean_text(description)
    
    # Clasificar
    start = time.time()
    es_aire_libre = air_classifier.classify_single(clean_desc, title)
    elapsed = time.time() - start
    
    print(f"📍 {title[:50]}...")
    print(f"   🌳 Aire libre: {es_aire_libre} ({elapsed:.2f}s)")
    print(f"   📝 Desc: {clean_desc[:60]}...\n")

## Paso 6: Probar extracción de edades

In [ ]:
print("👶 Probando extracción de edades...\n")

for act in actividades_prueba:
    title = act.get('title', '')
    description = act.get('description', '')
    
    # Limpiar texto
    clean_desc = clean_text(description)
    
    # Extraer edades
    start = time.time()
    edades = age_extractor.extract_ages(clean_desc, title)
    elapsed = time.time() - start
    
    print(f"📍 {title[:50]}...")
    print(f"   👶 Edad mínima: {edades['edad_minima']}")
    print(f"   👴 Edad máxima: {edades['edad_maxima']}")
    print(f"   ⏱️ Tiempo: {elapsed:.2f}s\n")

## Paso 7: Procesar actividad completa (todos los campos)

In [ ]:
print("🔬 Procesando actividad completa con todos los campos...\n")

# Seleccionar una actividad para análisis detallado
actividad = actividades_prueba[0]

print(f"📋 Actividad: {actividad.get('title')}")
print(f"📝 Descripción completa:")
print(f"{actividad.get('description', 'Sin descripción')}")
print("\n" + "="*60 + "\n")

# Preparar texto
title = actividad.get('title', '')
description = actividad.get('description', '')
clean_desc = clean_text(description)

print(f"🧹 Texto limpio ({len(clean_desc)} chars):")
print(f"{clean_desc[:200]}...\n")

# Procesar
print("🤖 Ejecutando modelos...\n")

# Clasificar aire libre
es_aire_libre = air_classifier.classify_single(clean_desc, title)
print(f"🌳 Es aire libre: {es_aire_libre}")

# Extraer edades
edades = age_extractor.extract_ages(clean_desc, title)
print(f"👶 Edad mínima: {edades['edad_minima']}")
print(f"👴 Edad máxima: {edades['edad_maxima']}")

# Resultado final
print("\n" + "="*60)
print("📊 RESULTADO FINAL:")
print("="*60)
resultado = {
    **actividad,
    'es_aire_libre': es_aire_libre,
    'edad_minima': edades['edad_minima'],
    'edad_maxima': edades['edad_maxima']
}
print(json.dumps({
    'title': resultado['title'],
    'es_aire_libre': resultado['es_aire_libre'],
    'edad_minima': resultado['edad_minima'],
    'edad_maxima': resultado['edad_maxima']
}, indent=2, ensure_ascii=False))

## Paso 8: Probar con texto personalizado

Prueba con tus propios textos para ver cómo funciona la extracción

In [ ]:
# Define tu propio texto de prueba
TEXTO_PRUEBA = """
Taller de pintura al aire libre en el parque del Retiro. 
Para niños y niñas de 6 a 12 años. 
Traer ropa cómoda y ganas de divertirse.
"""

TITULO_PRUEBA = "Taller de Pintura en el Retiro"

print(f"📝 Texto de prueba:")
print(f"{TEXTO_PRUEBA}\n")

# Limpiar
clean_test = clean_text(TEXTO_PRUEBA)

# Clasificar
es_aire_libre = air_classifier.classify_single(clean_test, TITULO_PRUEBA)
print(f"🌳 Es aire libre: {es_aire_libre}")

# Extraer edades
edades = age_extractor.extract_ages(clean_test, TITULO_PRUEBA)
print(f"👶 Edad mínima: {edades['edad_minima']}")
print(f"👴 Edad máxima: {edades['edad_maxima']}")

## Paso 9: Ejecutar script completo (opcional)

Descomenta para ejecutar el script completo con todas las actividades

In [ ]:
# Ejecutar script completo con límite de 10 actividades
# !python ../scripts/enrich_activities.py --limit 10 --output ../data/test_output.json

# Ver resultado
# if os.path.exists('../data/test_output.json'):
#     with open('../data/test_output.json', 'r') as f:
#         resultado = json.load(f)
#     print(f"✅ Procesadas {len(resultado)} actividades")
#     print("\nPrimer resultado:")
#     print(json.dumps(resultado[0], indent=2, ensure_ascii=False))

## 📊 Resumen y Estadísticas

In [ ]:
print("📈 ESTADÍSTICAS DEL MODELO")
print("="*60)

print("🤖 Modelos utilizados:")
print(f"   - Clasificación: {air_classifier.MODEL_NAME}")
print(f"   - QA Edades: {age_extractor.MODEL_NAME}")

print("\n⚙️ Configuración:")
print(f"   - Device: CPU")
print(f"   - Batch size: 32 (configurable)")

print("\n📋 Campos generados:")
print("   - es_aire_libre: booleano")
print("   - edad_minima: entero o null")
print("   - edad_maxima: entero o null")

print("\n💡 Notas:")
print("   - La primera ejecución descarga los modelos (~500MB)")
print("   - Tiempo estimado: 1-2 segundos por actividad")
print("   - Usa regex + IA para mejor precisión")